In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import math

from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import Perceptron
from sklearn.neural_network import MLPClassifier


In [ ]:
df=pd.read_csv("")

# distribution of target variable

sns.histplot(df['Loan Amount'],kde=True)

# for classification, use sns.countplot(x=df['data'])

plt.title("")
plt.xlabel("")
plt.ylabel("")
plt.show()

In [ ]:
#subplots

num_cols = df.select_dtypes(include='number').columns

plt.figure(figsize=(12, 8))

for i, col in enumerate(num_cols, 1):
    plt.subplot(3, 3, i)   # 3x3 grid (change if needed)
    sns.histplot(df[col], kde=True)
    plt.title(col)

plt.tight_layout()
plt.show()

In [ ]:
#ALTERNATIVE USE THIS

df.hist(bins=30,figsize=(10,10))

In [ ]:
# correlation matrix
corr = df.corr()

sns.heatmap(corr, annot=True, cmap='coolwarm')

plt.title("Correlation Matrix")
plt.show()

In [ ]:
df=df.dropna(subset=['Loan Amount'])

X=df.drop(['Loan Amount','Customer ID','Name'],axis=1)
y=df['Loan Amount']

X_train, X_test, y_train, y_test=train_test_split(X,y,test_size=0.2, random_state=42)

numeric_features=[col for col in X.columns if X[col].dtype!='object']
categorical_features=[col for col in X.columns if X[col].dtype=='object']

In [ ]:
numeric_transformer=Pipeline([('imputer',SimpleImputer(strategy='median')), ('scaler',StandardScaler())])
categorical_transformer=Pipeline([('imputer',SimpleImputer(strategy='most_frequent')), ('ohe',OneHotEncoder(handle_unknown='ignore'))])

preprocessor=ColumnTransformer(transformers=[('num',numeric_transformer,numeric_features),('cat',categorical_transformer,categorical_features)])

pipe=Pipeline([('preprocessor',preprocessor),('regression',KNeighborsRegressor())])

# ('classifier', SVC(kernel='rbf'))
# ('classifier', RandomForestClassifier())
# ('classifier', KNeighborsClassifier())
# ('regression', LinearRegression())
# ('regression',ElasticNet())
#('regression',KNeighborsRegressor())
#  ('regression', SVR())

param_grid={
    'regression__alpha':[0.1,1.0]
}
#scoring=['accuracy'] for classification
grid=GridSearchCV(pipe,param_grid,cv=5,scoring='r2',n_jobs=-1,return_train_score=True)
grid.fit(X_train,y_train)

grid.best_params_
grid.score(X_test, y_test)

In [ ]:
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error
)

y_pred = model.predict(X_test)

print("R2 Score:", r2_score(y_test, y_pred))
print("MSE:", mean_squared_error(y_test, y_pred))
print("RMSE:", mean_squared_error(y_test, y_pred, squared=False))
print("MAE:", mean_absolute_error(y_test, y_pred))

In [ ]:
from sklearn.preprocessing import label_binarize

def report(model, X_test, y_test, is_reg=False):
    import matplotlib.pyplot as plt

    plt.figure(figsize=(12, 5))

    if is_reg:
        from sklearn.metrics import PredictionErrorDisplay

        plt.subplot(1, 2, 1)
        PredictionErrorDisplay.from_estimator(model, X_test, y_test)
        plt.title("Actual vs Predicted")

        plt.subplot(1, 2, 2)
        PredictionErrorDisplay.from_estimator(model, X_test, y_test, kind="residual_vs_predicted")
        plt.title("Residuals")

    else:
        from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay
        from sklearn.preprocessing import label_binarize
        import numpy as np

        # Confusion Matrix
        plt.subplot(1, 2, 1)
        ConfusionMatrixDisplay.from_estimator(model, X_test, y_test)

        # ROC Curve
        plt.subplot(1, 2, 2)

        y_prob = model.predict_proba(X_test)
        classes = np.unique(y_test)
        y_test_bin = label_binarize(y_test, classes=classes)

        for i in range(len(classes)):
            RocCurveDisplay.from_predictions(
                y_test_bin[:, i],
                y_prob[:, i],
                ax=plt.gca()
            )

        plt.title("ROC Curve")

    plt.tight_layout()
    plt.show()

In [ ]:
report(grid, X_test, y_test, is_reg=False)   # classification
report(grid, X_test, y_test, is_reg=True)    # regression

## GRID SEARCH AND CROSS VALIDATION

In [ ]:
# ===============================
# 1. Imports
# ===============================
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score


# ===============================
# 2. Sample Data (replace with yours)
# ===============================
from sklearn.datasets import load_iris

data = load_iris()
X = data.data
y = data.target

# Train-test split (IMPORTANT)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


# ===============================
# 3. Pipeline
# ===============================
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', RandomForestClassifier(random_state=42))
])


# ===============================
# 4. Parameter Grid
# ===============================
param_grid = {
    'model__n_estimators': [50, 100],
    'model__max_depth': [3, 5]
}


# ===============================
# 5. Multiple Scoring
# ===============================
scoring = {
    'accuracy': 'accuracy',
    'f1': 'f1_macro',
    'precision': 'precision_macro',
    'recall': 'recall_macro'
}

# for regression, use r2


# ===============================
# 6. GridSearchCV
# ===============================
grid = GridSearchCV(
    pipe,
    param_grid,
    cv=5,
    scoring=scoring,
    refit='accuracy',  # choose best model based on accuracy
    n_jobs=-1,
    return_train_score=True
)


# ===============================
# 7. Train Model
# ===============================
grid.fit(X_train, y_train)


# ===============================
# 8. Convert CV Results to DataFrame
# ===============================
results = pd.DataFrame(grid.cv_results_)

print("\n=== All CV Results (first few rows) ===")
print(results.head())


# ===============================
# 9. Show Validation Scores
# ===============================
print("\n=== Validation Scores ===")
print(results[['mean_test_accuracy',
               'mean_test_f1',
               'mean_test_precision',
               'mean_test_recall']])


# ===============================
# 10. Show Training Scores
# ===============================
print("\n=== Training Scores ===")
print(results[['mean_train_accuracy',
               'mean_train_f1']])


# ===============================
# 11. Best Model Details
# ===============================
best_index = grid.best_index_

print("\n=== Best Parameters ===")
print(grid.best_params_)

print("\n=== Best CV Accuracy ===")
print(grid.best_score_)


# Extract best model scores
best_row = results.loc[best_index]

print("\n=== Best Model CV Scores ===")
print("Train Accuracy:", best_row['mean_train_accuracy'])
print("Validation Accuracy:", best_row['mean_test_accuracy'])
print("Validation F1:", best_row['mean_test_f1'])
print("Validation Precision:", best_row['mean_test_precision'])
print("Validation Recall:", best_row['mean_test_recall'])


# ===============================
# 12. Final Test Evaluation (UNSEEN DATA)
# ===============================
y_pred = grid.predict(X_test)

print("\n=== Final Test Performance ===")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred, average='macro'))
print("Precision:", precision_score(y_test, y_pred, average='macro'))
print("Recall:", recall_score(y_test, y_pred, average='macro'))


# ===============================
# 13. Predict New Data (Example)
# ===============================
sample = X_test[0].reshape(1, -1)

prediction = grid.predict(sample)

print("\n=== Sample Prediction ===")
print("Predicted class:", prediction)

## KNN GRAPH

import pandas as pd
import matplotlib.pyplot as plt

# Convert results
results = pd.DataFrame(grid.cv_results_)

# Extract values
neighbors = results['param_classification__n_neighbors']
accuracy = results['mean_test_score']

# Plot
plt.plot(neighbors, accuracy, marker='o')
plt.xlabel("Number of Neighbors (k)")
plt.ylabel("Accuracy")
plt.title("KNN: Neighbors vs Accuracy")
plt.show()

## Bagging

In [ ]:
from sklearn.ensemble import
 BaggingClassifier, AdaBoostClassifier, 
 GradientBoostingClassifier, StackingClassifier


bag_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('model', BaggingClassifier(estimator=DecisionTreeClassifier(), random_state=42))
])

bag_param = {
    'model__n_estimators': [10, 50],
    'model__max_samples': [0.5, 1.0],
    'model__max_features': [0.5, 1.0]
}

bag_grid = GridSearchCV(bag_pipe, bag_param, cv=5, scoring=scoring,
                        refit='accuracy', n_jobs=-1, return_train_score=True)

bag_grid.fit(X_train, y_train)

## Adaboost

In [ ]:
ada_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', AdaBoostClassifier(random_state=42))
])

ada_param = {
    'model__n_estimators': [50, 100],
    'model__learning_rate': [0.5, 1.0]
}

ada_grid = GridSearchCV(ada_pipe, ada_param, cv=5, scoring=scoring,
                        refit='accuracy', n_jobs=-1)

ada_grid.fit(X_train, y_train)

## GradientBoost

In [ ]:
gb_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', GradientBoostingClassifier(random_state=42))
])

gb_param = {
    'model__n_estimators': [50, 100],
    'model__learning_rate': [0.05, 0.1],
    'model__max_depth': [3, 5]
}

gb_grid = GridSearchCV(gb_pipe, gb_param, cv=5, scoring=scoring,
                       refit='accuracy', n_jobs=-1)

gb_grid.fit(X_train, y_train)

## Stacking

In [ ]:
stack_model = StackingClassifier(
    estimators=[
        ('svm', SVC(probability=True)),
        ('nb', GaussianNB()),
        ('dt', DecisionTreeClassifier())
    ],
    final_estimator=LogisticRegression()
)

stack_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', stack_model)
])

stack_param = {
    'model__final_estimator__C': [0.1, 1.0]
}

stack_grid = GridSearchCV(stack_pipe, stack_param, cv=5,
                          scoring=scoring, refit='accuracy', n_jobs=-1)

stack_grid.fit(X_train, y_train)

## PCA

In [ ]:
from sklearn.decomposition import PCA

# Without PCA
pipe1 = Pipeline([("prep", preprocessor), ("model", model)])

# With PCA

pipe2 = Pipeline([
            ("prep", preprocessor),
            ("pca", PCA(n_components=0.95)),
            ("model", model)
        ])

## Clustering

In [ ]:
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score,
    adjusted_rand_score,
    normalized_mutual_info_score
)

X = pd.read_csv("X.csv")
y = pd.read_csv("y.csv")

y = y.values.ravel()

In [ ]:
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=2))
])
X_pca = pipe.fit_transform(X)

## KMeans (Elbow Method)

In [ ]:
wcss = []
sil_scores = []
K = range(2, 9)

for k in K:
    kmeans=KMeans(n_clusters=k, random_state=42)
    labels=kmeans.fit_predict(X)
    wcss.append(kmeans.intertia_)
    sit_scores.append(silhouette_score(X,labels))

# Elbow Plot
plt.plot(K, wcss, marker='o')
plt.xlabel("k")
plt.ylabel("WCSS")
plt.title("Elbow Method")
plt.show()

# Silhouette Plot
plt.plot(K, sil_scores, marker='o')
plt.xlabel("k")
plt.ylabel("Silhouette Score")
plt.title("Silhouette Score vs k")
plt.show()



## Kmeans

In [ ]:
# Choose best k and do k means
kmeans = KMeans(n_clusters=3, random_state=42)
k_labels = kmeans.fit_predict(X)

## DBSCAN

In [ ]:
dbscan = DBSCAN(eps=1.5, min_samples=5)
d_labels = dbscan.fit_predict(X)

## Hierarchical (Agglomerative)

In [ ]:
hac = AgglomerativeClustering(n_clusters=3, linkage='ward')
h_labels = hac.fit_predict(X)

In [ ]:
def evaluate(name, labels):
    print(f"\n{name} Results:")

    print("Silhouette:", silhouette_score(X, labels))
    print("Davies-Bouldin:", davies_bouldin_score(X, labels))
    print("Calinski-Harabasz:", calinski_harabasz_score(X, labels))

    print("ARI:", adjusted_rand_score(y, labels))
    print("NMI:", normalized_mutual_info_score(y, labels))

In [ ]:
def plot_clusters(X_pca, labels, title):
    plt.scatter(X_pca[:, 0], X_pca[:, 1], c=labels, cmap='viridis')
    plt.title(title)
    plt.show()

plot_clusters(X_pca, k_labels, "KMeans")
plot_clusters(X_pca, d_labels, "DBSCAN")
plot_clusters(X_pca, h_labels, "Hierarchical")

## Kmeans evaluation if using pipeline

In [ ]:
# Get cluster labels
labels = pipe.fit_predict(X)

# Get transformed data (after preprocessing)
X_transformed = pipe.named_steps['preprocessor'].transform(X)

from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score
)

print("Silhouette Score:", silhouette_score(X_transformed, labels))
print("Davies-Bouldin Index:", davies_bouldin_score(X_transformed, labels))
print("Calinski-Harabasz Score:", calinski_harabasz_score(X_transformed, labels))
